# FT-Transformer

In [1]:
!python -m pip install --upgrade pip setuptools wheel

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [2]:
!pip install ipywidgets

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [3]:
!pip install rtdl_revisiting_models -q

In [4]:
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from rtdl_revisiting_models import FTTransformer

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
DATA_PATH = "FeatureA_Repeated"
OUTPUT_PATH = "Official_FTTransformer_FeatureA_Results"


os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

cuda


In [6]:
def get_feature_cols(df):
    return [
        col for col in df.columns
        if col not in ["userId", "movieId", "label"]
    ]

In [7]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [8]:
def compute_auc_from_scratch(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    sorted_indices = np.argsort(-y_score)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            * (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc


def compute_metrics_from_scratch(y_true, y_pred, y_score):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    auc = compute_auc_from_scratch(y_true, y_score)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [9]:
def stratified_sample_binary(df, sample_size, random_seed):
    if sample_size >= len(df):
        return df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    sample_ratio = sample_size / len(df)

    sampled_df = (
        df
        .groupby("label", group_keys=False)
        .apply(
            lambda x: x.sample(
                n=max(1, int(len(x) * sample_ratio)),
                random_state=random_seed
            )
        )
        .sample(frac=1, random_state=random_seed)
        .reset_index(drop=True)
    )

    return sampled_df

In [10]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)

    train_indices = []
    test_indices = []

    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)

        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [11]:
def train_official_ft_transformer(
    train_df,
    test_df,
    lr=1e-4,
    weight_decay=1e-5,
    batch_size=4096,
    epochs=3
):
    feature_cols = get_feature_cols(train_df)

    X_train = train_df[feature_cols].values
    y_train = train_df["label"].values

    X_test = test_df[feature_cols].values
    y_test = test_df["label"].values

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    train_dataset = TabularDataset(X_train, y_train)
    test_dataset = TabularDataset(X_test, y_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model = FTTransformer(
        n_cont_features=len(feature_cols),
        cat_cardinalities=[],
        d_out=1,
        **FTTransformer.get_default_kwargs()
    ).to(device)

    optimizer = model.make_default_optimizer()
    
    # Override default optimizer lr / weight_decay if needed
    for group in optimizer.param_groups:
        group["lr"] = lr
        group["weight_decay"] = weight_decay

    criterion = nn.BCEWithLogitsLoss()

    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X, None).squeeze(1)

            loss = criterion(logits, batch_y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1} loss:", round(total_loss, 4))

    model.eval()

    all_preds = []
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)

            logits = model(batch_X, None).squeeze(1)
            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).int()

            all_preds.extend(preds.detach().cpu().tolist())
            all_scores.extend(probs.detach().cpu().tolist())
            all_labels.extend(batch_y.detach().cpu().tolist())

    metrics = compute_metrics_from_scratch(
        all_labels,
        all_preds,
        all_scores
    )

    return metrics

In [12]:
TRAIN_SAMPLE_SIZE = 30000
TEST_SAMPLE_SIZE = 10000

LR_VALUES = [5e-5, 1e-4, 5e-4]
WEIGHT_DECAY_VALUES = [1e-5]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2

BATCH_SIZE = 1024
EPOCHS = 3

all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):
    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_A_train.csv")
    test_path = os.path.join(repeat_folder, "feature_A_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df = stratified_sample_binary(
        train_df,
        sample_size=TRAIN_SAMPLE_SIZE,
        random_seed=42 + repeat_id
    )

    test_df = stratified_sample_binary(
        test_df,
        sample_size=TEST_SAMPLE_SIZE,
        random_seed=100 + repeat_id
    )

    print("Outer train shape:", train_df.shape)
    print("Outer test shape:", test_df.shape)

    best_lr = None
    best_weight_decay = None
    best_mean_val_f1 = -1

    for lr in LR_VALUES:
        for weight_decay in WEIGHT_DECAY_VALUES:

            inner_f1_scores = []

            for inner_id in range(N_INNER_REPEATS):
                inner_train_df, val_df = stratified_split_from_scratch(
                    train_df,
                    label_col="label",
                    test_ratio=VALID_RATIO,
                    random_seed=2000 + repeat_id * 10 + inner_id
                )

                print(
                    f"Trying lr={lr}, weight_decay={weight_decay}, inner={inner_id + 1}"
                )

                val_metrics = train_official_ft_transformer(
                    train_df=inner_train_df,
                    test_df=val_df,
                    lr=lr,
                    weight_decay=weight_decay,
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS
                )

                inner_f1_scores.append(val_metrics["f1"])

            mean_val_f1 = np.mean(inner_f1_scores)
            std_val_f1 = np.std(inner_f1_scores, ddof=1)

            all_tuning_results.append({
                "outer_repeat": repeat_id,
                "lr": lr,
                "weight_decay": weight_decay,
                "mean_validation_f1": mean_val_f1,
                "std_validation_f1": std_val_f1
            })

            print("Mean validation F1:", round(mean_val_f1, 6))

            if mean_val_f1 > best_mean_val_f1:
                best_mean_val_f1 = mean_val_f1
                best_lr = lr
                best_weight_decay = weight_decay

    print("Best lr:", best_lr)
    print("Best weight_decay:", best_weight_decay)
    print("Best mean validation F1:", best_mean_val_f1)

    test_metrics = train_official_ft_transformer(
        train_df=train_df,
        test_df=test_df,
        lr=best_lr,
        weight_decay=best_weight_decay,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS
    )

    final_result = {
        "repeat": repeat_id,
        "best_lr": best_lr,
        "best_weight_decay": best_weight_decay,
        "batch_size": BATCH_SIZE,
        "best_mean_val_f1": best_mean_val_f1,
        **test_metrics
    }

    all_results.append(final_result)

    print("Accuracy :", round(test_metrics["accuracy"], 4))
    print("Precision:", round(test_metrics["precision"], 4))
    print("Recall   :", round(test_metrics["recall"], 4))
    print("F1       :", round(test_metrics["f1"], 4))
    print("AUC      :", round(test_metrics["auc"], 4))

Outer Repeat 01


/tmp/ipykernel_1729/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_1729/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 17)
Outer test shape: (9999, 17)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.0342
Epoch 2 loss: 8.5272
Epoch 3 loss: 7.9982
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 11.8186
Epoch 2 loss: 8.7069
Epoch 3 loss: 7.9857
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 12.6029
Epoch 2 loss: 8.9063
Epoch 3 loss: 8.1752
Mean validation F1: 0.84759
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 10.3336
Epoch 2 loss: 8.1134
Epoch 3 loss: 7.6287
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.6122
Epoch 2 loss: 8.1281
Epoch 3 loss: 7.6298
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 10.318
Epoch 2 loss: 7.988
Epoch 3 loss: 7.6608
Mean validation F1: 0.849839
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 9.7367
Epoch 2 loss: 7.5178
Epoch 3 loss: 7.3225
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.7435
Epoch 2 loss: 7.6881
Epoch 3 loss: 7.3397
Trying lr

/tmp/ipykernel_1729/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_1729/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 17)
Outer test shape: (9999, 17)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.9319
Epoch 2 loss: 8.5985
Epoch 3 loss: 7.955
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 12.0767
Epoch 2 loss: 8.7393
Epoch 3 loss: 8.1113
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 11.2054
Epoch 2 loss: 8.4591
Epoch 3 loss: 7.8514
Mean validation F1: 0.850374
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 10.3851
Epoch 2 loss: 8.0009
Epoch 3 loss: 7.648
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.3197
Epoch 2 loss: 8.0763
Epoch 3 loss: 7.6345
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 10.1308
Epoch 2 loss: 7.976
Epoch 3 loss: 7.4955
Mean validation F1: 0.851699
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 10.6784
Epoch 2 loss: 7.6068
Epoch 3 loss: 7.324
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.1746
Epoch 2 loss: 7.6228
Epoch 3 loss: 7.3437
Trying lr

...

Outer train shape: (29999, 17)
Outer test shape: (9999, 17)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.437
Epoch 2 loss: 8.6872
Epoch 3 loss: 8.0318
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 11.2319
Epoch 2 loss: 8.5605
Epoch 3 loss: 7.961
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 12.9656
Epoch 2 loss: 8.8575
Epoch 3 loss: 8.1218
Mean validation F1: 0.850334
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 10.4192
Epoch 2 loss: 8.0047
Epoch 3 loss: 7.5659
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.8036
Epoch 2 loss: 8.2266
Epoch 3 loss: 7.6372
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 10.4312
Epoch 2 loss: 8.0454
Epoch 3 loss: 7.6014
Mean validation F1: 0.847869
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 10.554
Epoch 2 loss: 7.6641
Epoch 3 loss: 7.293
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.4171
Epoch 2 loss: 7.6541
Epoch 3 loss: 7.4866
Trying lr

/tmp/ipykernel_1729/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_1729/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 17)
Outer test shape: (9999, 17)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.444
Epoch 2 loss: 8.5532
Epoch 3 loss: 7.9677
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.8482
Epoch 2 loss: 8.4032
Epoch 3 loss: 7.8385
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 11.7867
Epoch 2 loss: 8.7119
Epoch 3 loss: 8.0471
Mean validation F1: 0.852475
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 10.3029
Epoch 2 loss: 8.0709
Epoch 3 loss: 7.6482
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.3288
Epoch 2 loss: 7.9337
Epoch 3 loss: 7.5161
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 10.7846
Epoch 2 loss: 8.0739
Epoch 3 loss: 7.6157
Mean validation F1: 0.852869
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 10.0271
Epoch 2 loss: 7.6263
Epoch 3 loss: 7.4391
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.3158
Epoch 2 loss: 7.5356
Epoch 3 loss: 7.2946
Trying

/tmp/ipykernel_1729/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_1729/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 17)
Outer test shape: (9999, 17)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 11.6516
Epoch 2 loss: 8.4974
Epoch 3 loss: 7.893
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 11.3422
Epoch 2 loss: 8.4144
Epoch 3 loss: 7.9029
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 11.3587
Epoch 2 loss: 8.574
Epoch 3 loss: 8.0025
Mean validation F1: 0.852229
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 10.5874
Epoch 2 loss: 7.9766
Epoch 3 loss: 7.541
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.3684
Epoch 2 loss: 7.969
Epoch 3 loss: 7.5139
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 10.6213
Epoch 2 loss: 7.9125
Epoch 3 loss: 7.5664
Mean validation F1: 0.856033
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 10.6138
Epoch 2 loss: 7.5839
Epoch 3 loss: 7.342
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 10.8739
Epoch 2 loss: 7.5784
Epoch 3 loss: 7.3012
Trying lr=

In [13]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.DataFrame(all_tuning_results)

results_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureA_repeated_results.csv"
)

tuning_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureA_tuning_results.csv"
)

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
Official_FTTransformer_FeatureA_Results/FTTransformer_FeatureA_repeated_results.csv
Saved tuning results to:
Official_FTTransformer_FeatureA_Results/FTTransformer_FeatureA_tuning_results.csv


,repeat,best_lr,best_weight_decay,batch_size,best_mean_val_f1,accuracy,precision,recall,f1,auc,tp,tn,fp,fn
0,1,0.00010,0.00001,1024,0.849839,0.618762,0.649583,0.514909,0.574459,0.643940,2573,3614,1388,2424
1,2,0.00050,0.00001,1024,0.857933,0.608061,0.644970,0.479888,0.550316,0.638480,2398,3682,1320,2599
2,3,0.00010,0.00001,1024,0.848972,0.636964,0.674407,0.528917,0.592867,0.661162,2643,3726,1276,2354
3,4,0.00010,0.00001,1024,0.856333,0.636364,0.650787,0.587753,0.617666,0.653776,2937,3426,1576,2060
4,5,0.00050,0.00001,1024,0.853621,0.606861,0.638514,0.491695,0.555568,0.640270,2457,3611,1391,2540
5,6,0.00050,0.00001,1024,0.850071,0.596360,0.643219,0.431859,0.516762,0.601930,2158,3805,1197,2839
6,7,0.00010,0.00001,1024,0.847368,0.627363,0.636344,0.593556,0.614206,0.644845,2966,3307,1695,2031
7,8,0.00005,0.00001,1024,0.850334,0.615662,0.639845,0.528317,0.578757,0.645114,2640,3516,1486,2357
8,9,0.00010,0.00001,1024,0.852869,0.630263,0.653810,0.552932,0.599154,0.652064,2763,3539,1463,2234
9,10,0.00010,0.00001,1024,0.856033,0.637064,0.658187,0.569542,0.610664,0.655462,2846,3524,1478,2151


In [14]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureA_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

summary_df

,metric,mean,std,standard_error
0,accuracy,0.621372,0.014460,0.004573
1,precision,0.648967,0.011354,0.003590
2,recall,0.527937,0.050841,0.016077
3,f1,0.581042,0.032592,0.010306
4,auc,0.643704,0.016336,0.005166


In [15]:
# Best learning rate frequency for FT-Transformer

best_lr_frequency = (
    results_df["best_lr"]
    .value_counts()
    .reset_index()
)

best_lr_frequency.columns = ["learning_rate", "frequency"]

best_lr_frequency_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureA_best_lr_frequency.csv"
)

best_lr_frequency.to_csv(
    best_lr_frequency_path,
    index=False,
    encoding="utf-8-sig"
)


best_lr_frequency

,learning_rate,frequency
0,0.00010,6
1,0.00050,3
2,0.00005,1
